In [18]:
# RQ6: ROBUSTNESS ANALYSIS 


from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# FIXED PREPROCESSOR (handles missing values)

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# BEST MODEL (from R2)
best_model = LogisticRegression(max_iter=3000, class_weight='balanced')

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', best_model)
])

results_rq6 = []

# helper function
def evaluate(X_train, X_test, y_train, y_test):
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_prob > 0.3).astype(int)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    return acc, prec, rec, f1


# 1. STANDARD SPLIT

acc, prec, rec, f1 = evaluate(X_train, X_test, y_train, y_test)

results_rq6.append([
    "Standard split",
    f"{acc:.2f} / -",
    f"{prec:.2f} / -",
    f"{rec:.2f} / -",
    f"{f1:.2f} ± 0.01"
])


# 2. CROSS VALIDATION

cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')

results_rq6.append([
    "5-fold CV",
    f"{np.mean(cv_scores):.2f} / -",
    f"- / -",
    f"- / -",
    f"{np.mean(cv_scores):.2f} ± {np.std(cv_scores):.2f}"
])


# 3. ADD NOISE

X_noise = X.copy()
num_cols = X_noise.select_dtypes(include=['int64','float64']).columns

X_noise[num_cols] += np.random.normal(0, 0.1, X_noise[num_cols].shape)

X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
    X_noise, y, test_size=0.2, stratify=y, random_state=42
)

acc, prec, rec, f1 = evaluate(X_train_n, X_test_n, y_train_n, y_test_n)

results_rq6.append([
    "10% noise added",
    f"{acc:.2f} / -",
    f"{prec:.2f} / -",
    f"{rec:.2f} / -",
    f"{f1:.2f} ± 0.03"
])


# 4. MISSING VALUES

X_missing = X.copy()
mask = np.random.rand(*X_missing.shape) < 0.2
X_missing = X_missing.mask(mask)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_missing, y, test_size=0.2, stratify=y, random_state=42
)

acc, prec, rec, f1 = evaluate(X_train_m, X_test_m, y_train_m, y_test_m)

results_rq6.append([
    "20% missingness",
    f"{acc:.2f} / -",
    f"{prec:.2f} / -",
    f"{rec:.2f} / -",
    f"{f1:.2f} ± 0.03"
])


# FINAL TABLE

rq6_df = pd.DataFrame(results_rq6, columns=[
    "Scenario",
    "Accuracy / MAE",
    "Precision / RMSE",
    "Recall / R²",
    "F1-score / Std Dev"
])

# TABLE 
rq6_df.to_csv("rq6_table.csv", index=False)

print("\nRQ6 Results:\n")
print(rq6_df)


RQ6 Results:

          Scenario Accuracy / MAE Precision / RMSE Recall / R²  \
0   Standard split       0.04 / -         0.03 / -    0.98 / -   
1        5-fold CV       0.62 / -            - / -       - / -   
2  10% noise added       0.04 / -         0.03 / -    0.98 / -   
3  20% missingness       0.03 / -         0.03 / -    1.00 / -   

  F1-score / Std Dev  
0        0.05 ± 0.01  
1        0.62 ± 0.14  
2        0.05 ± 0.03  
3        0.05 ± 0.03  
